In [ ]:
# Figure 5A–B: Anatomy-matched neurotransmitter proximity

from pathlib import Path
import json
import io
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize

data_dir = Path('data')
data = h5py.File(data_dir / 'fig5ab.h5', 'r')
out_dir = Path('fig5_outputs')
out_dir.mkdir(parents=True, exist_ok=True)

n_perm = 10000
seed = 20260821
hist_bin_um = 0.01
hist_max_um = 60.0
perm_batch = 250
nt_order = ['ACH', 'GABA', 'GLUT', 'DA', 'SER', 'OCT']
nt_colors = {'ACH': '#2020df', 'DA': '#963939', 'GABA': '#df2020', 'GLUT': '#df9c20', 'OCT': '#107010', 'SER': '#701070'}
superregions = {
    'OL': ['ME_L', 'ME_R', 'AME_L', 'AME_R', 'LO_L', 'LO_R', 'LOP_L', 'LOP_R'],
    'MB': ['MB_CA_R', 'MB_CA_L', 'MB_PED_L', 'MB_PED_R', 'MB_VL_R', 'MB_VL_L', 'MB_ML_R', 'MB_ML_L'],
    'CX': ['FB', 'EB', 'PB', 'NO'],
    'LX': ['BU_L', 'BU_R', 'LAL_R', 'LAL_L', 'GA_R', 'GA_L'],
    'VLNP': ['AOTU_R', 'AOTU_L', 'PVLP_L', 'AVLP_L', 'PVLP_R', 'AVLP_R', 'PLP_L', 'PLP_R', 'WED_L', 'WED_R'],
    'LH': ['LH_L', 'LH_R'],
    'SNP': ['SLP_L', 'SLP_R', 'SIP_L', 'SIP_R', 'SMP_R', 'SMP_L'],
    'INP': ['CRE_L', 'CRE_R', 'ICL_L', 'SCL_L', 'ICL_R', 'SCL_R', 'IB_L', 'IB_R', 'ATL_R', 'ATL_L'],
    'AL': ['AL_L', 'AL_R'],
    'VMNP': ['VES_L', 'VES_R', 'EPA_R', 'EPA_L', 'GOR_L', 'GOR_R', 'SPS_L', 'SPS_R', 'IPS_R', 'IPS_L'],
    'PENP': ['SAD', 'AMMC_R', 'AMMC_L', 'FLA_R', 'FLA_L', 'CAN_R', 'CAN_L', 'PRW'],
    'GNG': ['GNG'],
    'OCG': ['OCG'],
}
super_order = list(superregions)
region_to_super = {region: super_region for super_region, regions in superregions.items() for region in regions}

plt.rcParams.update({'font.family': 'Arial', 'font.size': 7.5, 'axes.labelsize': 7.5, 'axes.titlesize': 8.5, 'xtick.labelsize': 7, 'ytick.labelsize': 7.5, 'axes.linewidth': 0.8, 'xtick.major.width': 0.8, 'ytick.major.width': 0.8, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none'})

def bh_fdr(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    q = np.full(len(pvalues), np.nan, dtype=float)
    ok = np.isfinite(pvalues)
    if not ok.any():
        return q
    x = pvalues[ok]
    order = np.argsort(x)
    ranked = x[order]
    m = len(ranked)
    adjusted = ranked * m / np.arange(1, m + 1, dtype=float)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    back = np.empty(m, dtype=float)
    back[order] = adjusted
    q[ok] = back
    return q

# Regional contact-distance distributions
manifest = json.loads(data.attrs['manifest'])
available_regions = set(manifest['regions'])
nt_groups = {tuple(x) for x in manifest['nt_groups']}
edges = np.arange(0.0, hist_max_um + hist_bin_um, hist_bin_um, dtype=np.float64)
n_bins = len(edges) - 1

def build_superregion_histograms():
    hist_matrix = np.zeros((len(super_order), n_bins), dtype=np.int64)
    exact_counts = np.zeros(len(super_order), dtype=np.int64)
    for i, sr in enumerate(super_order):
        for region in superregions[sr]:
            if region not in available_regions:
                continue
            values = data[f'contacts/all/{region}'][:]
            values = values[np.isfinite(values) & (values >= 0) & (values < hist_max_um)]
            hist_matrix[i] += np.histogram(values, bins=edges)[0]
            exact_counts[i] += len(data[f'contacts/all/{region}'])
    return hist_matrix, exact_counts

all_hist_matrix, all_counts = build_superregion_histograms()
all_hist = {sr: all_hist_matrix[i] for i, sr in enumerate(super_order)}
all_count = {sr: int(all_counts[i]) for i, sr in enumerate(super_order)}
total_all_contacts = int(np.sum(all_counts))
count_rows = []
for nt in nt_order:
    nt_total = 0
    for sr in super_order:
        regions = [region for region in superregions[sr] if region in available_regions]
        n_nt = 0
        for region in regions:
            if (region, nt) not in nt_groups:
                continue
            n_nt += len(data[f'contacts/nt/{region}__{nt}'])
        n_all = all_count[sr]
        nt_total += n_nt
        count_rows.append({'nt_type': nt, 'super_region': sr, 'n_nt_contacts': int(n_nt), 'n_all_contacts': int(n_all)})
counts_df = pd.DataFrame(count_rows)
counts_df['nt_fraction_within_class'] = counts_df['n_nt_contacts'] / counts_df.groupby('nt_type')['n_nt_contacts'].transform('sum')
counts_df['all_contact_fraction'] = counts_df['n_all_contacts'] / total_all_contacts
counts_df['occupancy_enrichment'] = np.where(counts_df['all_contact_fraction'] > 0, counts_df['nt_fraction_within_class'] / counts_df['all_contact_fraction'], np.nan)
observed_table = pd.read_csv(io.BytesIO(data['tables/observed'][:].tobytes()))
observed_table['nt_type'] = observed_table['nt_type'].astype(str).str.upper()

def probability_hist_quantile(probability_hist, q):
    h = np.asarray(probability_hist, dtype=np.float64)
    total = float(h.sum())
    if total <= 0:
        return np.nan
    h = h / total
    cumulative = np.cumsum(h)
    i = int(np.searchsorted(cumulative, q, side='left'))
    i = min(i, len(h) - 1)
    before = cumulative[i - 1] if i > 0 else 0.0
    p = h[i]
    if p <= 0:
        return float(edges[i])
    fraction = float(np.clip((q - before) / p, 0, 1))
    return float(edges[i] + fraction * hist_bin_um)

def rank_value_from_hist_rows(cumulative, counts, rank):
    batch_n = counts.shape[0]
    rows = np.arange(batch_n)
    idx = np.argmax(cumulative >= rank, axis=1)
    before = np.where(idx > 0, cumulative[rows, idx - 1], 0)
    in_bin = counts[rows, idx]
    fraction = (rank - before - 0.5) / in_bin
    fraction = np.clip(fraction, 0.0, 1.0)
    return edges[idx] + fraction * hist_bin_um

def medians_from_hist_rows(sampled_hist, total_sample_n):
    sampled_hist = np.asarray(sampled_hist, dtype=np.int64)
    cumulative = np.cumsum(sampled_hist, axis=1)
    if total_sample_n % 2:
        rank = (total_sample_n + 1) // 2
        return rank_value_from_hist_rows(cumulative, sampled_hist, rank)
    rank1 = total_sample_n // 2
    rank2 = rank1 + 1
    x1 = rank_value_from_hist_rows(cumulative, sampled_hist, rank1)
    x2 = rank_value_from_hist_rows(cumulative, sampled_hist, rank2)
    return 0.5 * (x1 + x2)

def composition_expected_hist(nt_counts):
    total_nt = int(sum(nt_counts.values()))
    mixture = np.zeros(n_bins, dtype=np.float64)
    for sr in super_order:
        n_nt = int(nt_counts[sr])
        if n_nt <= 0:
            continue
        population_hist = all_hist[sr].astype(np.float64)
        population_n = float(population_hist.sum())
        weight = n_nt / total_nt
        mixture += weight * (population_hist / population_n)
    return mixture

def stratified_null_medians(nt, nt_counts, rng):
    total_sample_n = int(sum(nt_counts.values()))
    null = np.empty(n_perm, dtype=np.float64)
    for start in range(0, n_perm, perm_batch):
        stop = min(start + perm_batch, n_perm)
        batch_n = stop - start
        pooled = np.zeros((batch_n, n_bins), dtype=np.int64)
        for sr in super_order:
            n_draw = int(nt_counts[sr])
            if n_draw <= 0:
                continue
            population_hist = all_hist[sr]
            population_n = int(population_hist.sum())
            if n_draw == population_n:
                pooled += population_hist[None, :]
            else:
                draw = rng.multivariate_hypergeometric(colors=population_hist, nsample=n_draw, size=batch_n)
                pooled += draw
        null[start:stop] = medians_from_hist_rows(pooled, total_sample_n)
    return null

# Anatomy-matched whole-brain null
result_rows = []
for nt_index, nt in enumerate(nt_order):
    nt_counts_df = counts_df.loc[counts_df['nt_type'] == nt].set_index('super_region')
    nt_counts = {sr: int(nt_counts_df.loc[sr, 'n_nt_contacts']) for sr in super_order}
    total_nt = int(sum(nt_counts.values()))
    observed_row_df = observed_table.loc[observed_table['nt_type'] == nt]
    observed_row = observed_row_df.iloc[0]
    observed_median = float(observed_row['observed_median_um'])
    expected_hist = composition_expected_hist(nt_counts)
    expected_median = probability_hist_quantile(expected_hist, 0.5)
    rng = np.random.default_rng(seed + 1009 * nt_index)
    null = stratified_null_medians(nt, nt_counts, rng)
    null_median = float(np.median(null))
    null_mean = float(np.mean(null))
    null_sd = float(np.std(null, ddof=1))
    null_low = float(np.quantile(null, 0.025))
    null_high = float(np.quantile(null, 0.975))
    lower = (1 + np.count_nonzero(null <= observed_median)) / (n_perm + 1)
    upper = (1 + np.count_nonzero(null >= observed_median)) / (n_perm + 1)
    p_two = min(1.0, 2.0 * min(lower, upper))
    percentile = 100.0 * (np.count_nonzero(null < observed_median) + 0.5 * np.count_nonzero(null == observed_median)) / n_perm
    effect = observed_median - null_median
    result_rows.append({
        'analysis_level': 'whole_brain_composition_matched', 'nt_type': nt, 'n_all_synapses': total_all_contacts,
        'n_nt_synapses': total_nt, 'n_superregions_with_nt': int(sum((n > 0 for n in nt_counts.values()))), 'observed_median_um': observed_median,
        'composition_expected_median_um': expected_median, 'null_median_um': null_median, 'null_mean_um': null_mean,
        'null_sd_um': null_sd, 'null_ci_low_um': null_low, 'null_ci_high_um': null_high,
        'observed_minus_null_um': effect, 'null_percentile': percentile, 'p_closer': lower,
        'p_farther': upper, 'p_two_sided': p_two
    })
    print(f'{nt:>5}: n={total_nt:,}; observed={observed_median:.4f}; composition-null={null_median:.4f}; effect={effect:+.4f} µm; P={p_two:.6g}')
result = pd.DataFrame(result_rows)
result['q_fdr'] = bh_fdr(result['p_two_sided'].to_numpy(float))
result['significant_fdr_0_05'] = result['q_fdr'] < 0.05
result['direction'] = np.where(result['observed_minus_null_um'] < 0, 'closer', np.where(result['observed_minus_null_um'] > 0, 'farther', 'no difference'))
result['_rank'] = result['nt_type'].map({nt: i for i, nt in enumerate(nt_order)})
result = result.sort_values('_rank').drop(columns=['_rank']).reset_index(drop=True)

# Whole-brain effect plot
def draw_panel_a(data):
    data = data.set_index('nt_type').reindex(nt_order).reset_index().dropna(subset=['observed_minus_null_um']).copy()
    data['null_low_centered'] = data['null_ci_low_um'] - data['null_median_um']
    data['null_high_centered'] = data['null_ci_high_um'] - data['null_median_um']
    y = np.arange(len(data))
    fig, ax = plt.subplots(figsize=(3.6, 2.65))
    ax.axvline(0, color='0.45', linestyle=(0, (3, 2)), linewidth=0.8, zorder=0)
    for i, row in data.iterrows():
        nt = row['nt_type']
        color = nt_colors[nt]
        effect = float(row['observed_minus_null_um'])
        ax.plot([float(row['null_low_centered']), float(row['null_high_centered'])], [i, i], color='0.62', linewidth=2.0, solid_capstyle='round', zorder=2)
        ax.plot([0, effect], [i, i], color=color, linewidth=0.8, alpha=0.55, zorder=1)
        ax.scatter(effect, i, s=42, facecolor=color, edgecolor='white', linewidth=0.65, zorder=4)
    ax.set_yticks(y)
    ax.set_yticklabels(data['nt_type'])
    ax.invert_yaxis()
    ax.set_xlabel('Observed − anatomy-matched null median distance (µm)')
    ax.set_ylabel('')
    values_for_limit = np.concatenate([data['observed_minus_null_um'].to_numpy(float), data['null_low_centered'].to_numpy(float), data['null_high_centered'].to_numpy(float)])
    max_abs = max(float(np.nanmax(np.abs(values_for_limit))), 0.25)
    limit = max_abs * 1.22
    ax.set_xlim(-limit, limit)
    ax.text(0.01, 1.035, 'Closer than expected', transform=ax.transAxes, ha='left', va='bottom', fontsize=6.5, color='0.35')
    ax.text(0.99, 1.035, 'Farther than expected', transform=ax.transAxes, ha='right', va='bottom', fontsize=6.5, color='0.35')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.tick_params(axis='y', left=False, length=0, pad=4)
    ax.tick_params(axis='x', direction='out', pad=2.5)
    ax.grid(False)
    offset = 0.025 * (2 * limit)
    for i, row in data.iterrows():
        effect = float(row['observed_minus_null_um'])
        if effect >= 0:
            x = effect + offset
            ha = 'left'
        else:
            x = effect - offset
            ha = 'right'
        ax.text(x, i, f'{effect:+.2f}', ha=ha, va='center', fontsize=6.5, color='0.25')
    ax.text(-0.16, 1.08, 'A', transform=ax.transAxes, fontsize=11, fontweight='bold', ha='left', va='top')
    fig.tight_layout()
    fig.savefig(out_dir / 'fig5a.svg', bbox_inches='tight')
    plt.show()
draw_panel_a(result)
print(result[['nt_type', 'n_nt_synapses', 'observed_median_um', 'null_median_um', 'observed_minus_null_um', 'p_two_sided', 'q_fdr']].to_string(index=False))

# Neurotransmitter-specific regional proximity maps
# Neuropil super-regions are colored by observed minus anatomy-matched null median distance.

nt_display = {nt: nt for nt in nt_order}

robust_color_percentile = 90.0
min_color_limit_um = 0.75
signed_power_gamma = 0.62
sig_opacity = 0.97
non_sig_opacity = 0.84
missing_opacity = 0.16
atlas_ambient = 0.92
atlas_diffuse = 0.08
atlas_specular = 0.0
atlas_window_size = (1100, 760)
atlas_camera_direction = np.array([0.22, -1.0, 0.18], dtype=float)
atlas_camera_distance_multiplier = 2.45
atlas_parallel_scale_multiplier = 0.5
atlas_crop_pad_px = 18
effect_cmap = LinearSegmentedColormap.from_list('proximity_effect_robust', ['#1F5A85', '#5F93B6', '#A9C6D8', '#D8D8D5', '#E8B9AA', '#CF735F', '#9A3232'], N=256)
plt.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'], 'font.size': 7.2, 'axes.labelsize': 7.5, 'xtick.labelsize': 6.8, 'ytick.labelsize': 7.2, 'axes.linewidth': 0.75, 'xtick.major.width': 0.75, 'ytick.major.width': 0.75, 'xtick.major.size': 3.0, 'ytick.major.size': 3.0, 'svg.fonttype': 'none', 'pdf.fonttype': 42, 'ps.fonttype': 42})

# Regional anatomical rendering
class SignedPowerNorm(Normalize):
    """Symmetric power transform around zero; gamma < 1 expands small effects."""

    def __init__(self, limit, gamma=0.62, clip=True):
        self.limit = float(limit)
        self.gamma = float(gamma)
        super().__init__(vmin=-self.limit, vmax=self.limit, clip=clip)

    def __call__(self, value, clip=None):
        arr = np.ma.asarray(value, dtype=float)
        x = np.clip(arr / self.limit, -1.0, 1.0)
        y = 0.5 + 0.5 * np.sign(x) * np.power(np.abs(x), self.gamma)
        return np.ma.array(y, mask=np.ma.getmask(arr))

    def inverse(self, value):
        y = np.asarray(value, dtype=float)
        z = 2.0 * y - 1.0
        return np.sign(z) * np.power(np.abs(z), 1.0 / self.gamma) * self.limit

def crop_white_margin(image, pad=18, threshold=248):
    image = np.asarray(image)
    rgb = image[..., :3]
    foreground = np.any(rgb < threshold, axis=2)
    yy, xx = np.where(foreground)
    y0 = max(0, int(yy.min()) - pad)
    y1 = min(image.shape[0], int(yy.max()) + pad + 1)
    x0 = max(0, int(xx.min()) - pad)
    x1 = min(image.shape[1], int(xx.max()) + pad + 1)
    return image[y0:y1, x0:x1]
super_result = pd.read_csv(io.BytesIO(data['tables/regional'][:].tobytes()))
super_result = super_result[super_result['super_region'].isin(super_order) & super_result['nt_type'].isin(nt_order)].copy()

def load_neuropil_meshes():
    needed = sorted((region for region in available_regions if region in region_to_super))
    meshes = {}
    for region in needed:
        mesh = data[f'meshes/{region}']
        vertices = np.asarray(mesh['vertices'][:], dtype=np.float64)
        faces = np.asarray(mesh['faces'][:], dtype=np.int64)
        meshes[region] = {'vertices': vertices, 'faces': faces}
    return meshes

def build_pyvista_superregion_meshes(neuropils):
    import pyvista as pv
    all_raw = np.vstack([neuropils[r]['vertices'] for r in neuropils])
    original_center = (all_raw.min(axis=0) + all_raw.max(axis=0)) / 2.0

    def transform_vertices(vertices_nm):
        centered = np.asarray(vertices_nm, dtype=np.float64) - original_center
        return np.column_stack([centered[:, 0], centered[:, 2], -centered[:, 1]]) / 1000.0
    region_poly = {}
    for region, data in neuropils.items():
        verts = transform_vertices(data['vertices'])
        faces = np.asarray(data['faces'], dtype=np.int64)
        vtk_faces = np.column_stack([np.full(len(faces), 3, dtype=np.int64), faces]).ravel()
        region_poly[region] = pv.PolyData(verts, vtk_faces)
    super_meshes = {}
    for sr in super_order:
        parts = [region_poly[r] for r in superregions[sr] if r in region_poly]
        if not parts:
            continue
        merged = parts[0].copy()
        for part in parts[1:]:
            merged = merged.merge(part, merge_points=False)
        super_meshes[sr] = merged
    context_parts = list(super_meshes.values())
    context = context_parts[0].copy()
    for part in context_parts[1:]:
        context = context.merge(part, merge_points=False)
    return (pv, super_meshes, context)

def calculate_atlas_norm():
    effects = pd.to_numeric(super_result['observed_minus_null_um'], errors='coerce').to_numpy(float)
    effects = effects[np.isfinite(effects)]
    abs_effects = np.abs(effects)
    raw_max = float(np.max(abs_effects))
    effect_limit = max(float(np.percentile(abs_effects, robust_color_percentile)), min_color_limit_um)
    norm = SignedPowerNorm(limit=effect_limit, gamma=signed_power_gamma, clip=True)
    return (norm, effect_limit, raw_max)

def render_nt_brain(nt, pv, super_meshes, context_mesh, norm):
    plotter = pv.Plotter(off_screen=True, window_size=atlas_window_size)
    plotter.set_background('white')
    plotter.enable_anti_aliasing('ssaa')
    table = super_result.loc[super_result['nt_type'] == nt].set_index('super_region')
    for sr in super_order:
        if sr not in super_meshes:
            continue
        if sr not in table.index:
            plotter.add_mesh(super_meshes[sr], color='#D8D8D5', opacity=missing_opacity, smooth_shading=False, ambient=1.0, diffuse=0.0, specular=0.0)
            continue
        row = table.loc[sr]
        effect = float(row['observed_minus_null_um'])
        q = float(row['q_fdr'])
        rgba = effect_cmap(norm(effect))
        rgb = tuple(rgba[:3])
        opacity = sig_opacity if q < 0.05 else non_sig_opacity
        plotter.add_mesh(super_meshes[sr], color=rgb, opacity=opacity, smooth_shading=False, ambient=atlas_ambient, diffuse=atlas_diffuse, specular=atlas_specular)
    bounds = np.asarray(context_mesh.bounds, dtype=float)
    mins = np.array([bounds[0], bounds[2], bounds[4]])
    maxs = np.array([bounds[1], bounds[3], bounds[5]])
    center = (mins + maxs) / 2.0
    span = maxs - mins
    diagonal = float(np.linalg.norm(span))
    direction = atlas_camera_direction / np.linalg.norm(atlas_camera_direction)
    camera_pos = center + direction * diagonal * atlas_camera_distance_multiplier
    plotter.camera_position = [tuple(camera_pos), tuple(center), (0.0, 0.0, 1.0)]
    plotter.enable_parallel_projection()
    plotter.camera.parallel_scale = max(span[0], span[2]) * atlas_parallel_scale_multiplier
    image = plotter.screenshot(return_img=True, transparent_background=False)
    plotter.close()
    return crop_white_margin(np.asarray(image), pad=atlas_crop_pad_px)

def prepare_atlas():
    neuropils = load_neuropil_meshes()
    pv, super_meshes, context_mesh = build_pyvista_superregion_meshes(neuropils)
    norm, effect_limit, raw_max = calculate_atlas_norm()
    images = {}
    for nt in nt_order:
        print(f'Rendering anatomical atlas: {nt}')
        images[nt] = render_nt_brain(nt, pv, super_meshes, context_mesh, norm)
    return (images, norm, effect_limit, raw_max)

def draw_panel_b(fig, subplot_spec, images, norm, effect_limit, raw_max, add_letter=True):
    sub = subplot_spec.subgridspec(2, 3, wspace=0.025, hspace=0.08)
    axes = []
    for idx, nt in enumerate(nt_order):
        ax = fig.add_subplot(sub[idx // 3, idx % 3])
        axes.append(ax)
        ax.imshow(images[nt])
        ax.set_axis_off()
        ax.text(0.02, 0.96, nt_display[nt], transform=ax.transAxes, ha='left', va='top', fontsize=8.4, fontweight='bold', color=nt_colors[nt])
    axes[0].text(0.0, 1.1, 'Regional origin of NT-specific tracheal proximity', transform=axes[0].transAxes, ha='left', va='bottom', fontsize=9.0, fontweight='bold', color='0.12')
    if add_letter:
        axes[0].text(-0.12, 1.12, 'B', transform=axes[0].transAxes, ha='left', va='bottom', fontsize=11, fontweight='bold')
    sm = plt.cm.ScalarMappable(norm=norm, cmap=effect_cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes, orientation='horizontal', fraction=0.04, pad=0.035, aspect=38, extend='both' if raw_max > effect_limit else 'neither')
    ticks = np.array([-effect_limit, -0.5 * effect_limit, -0.2 * effect_limit, 0.0, 0.2 * effect_limit, 0.5 * effect_limit, effect_limit])
    cbar.set_ticks(ticks)
    cbar.set_ticklabels([f'{x:.2f}' if abs(x) < 1 else f'{x:.1f}' for x in ticks])
    cbar.set_label('Observed − super-region null median distance (µm)', labelpad=4)
    cbar.ax.tick_params(labelsize=6.8, length=2.3, width=0.6, pad=2)
    cbar.outline.set_linewidth(0.6)
    cbar.ax.text(0.0, -1.85, 'closer', transform=cbar.ax.transAxes, ha='left', va='top', fontsize=6.5, color='0.35')
    cbar.ax.text(1.0, -1.85, 'farther', transform=cbar.ax.transAxes, ha='right', va='top', fontsize=6.5, color='0.35')
    return axes
atlas_images, atlas_norm, atlas_effect_limit, atlas_raw_max = prepare_atlas()
fig_b = plt.figure(figsize=(7.2, 4.55), facecolor='white')
gs_b = fig_b.add_gridspec(1, 1)
draw_panel_b(fig_b, gs_b[0, 0], atlas_images, atlas_norm, atlas_effect_limit, atlas_raw_max, add_letter=False)
fig_b.subplots_adjust(left=0.025, right=0.985, bottom=0.15, top=0.92)
fig_b.savefig(out_dir / 'fig5b.svg', bbox_inches='tight')
plt.show()